# Fine-tune LSTM64-delay-8 on hand-switch negatives

Based on `training_lstm.ipynb`: the same delayed targets, positive-window amplitude augmentation, weighted soft cross-entropy, AdamW, cosine scheduler, and early stopping. This experiment **loads the saved best model**, rather than initializing random weights. Run cells in order from the repository root or `main_notebooks/`.

- Source: `testing_checkpoints/lstm64d8/best.pt`; one LSTM layer, 64 hidden units, delay 8. Architecture, seed, preprocessing, and loss settings are read from that checkpoint.
- Data: all old and new recordings in `data_05_10_26/train_data` and `data_05_10_26/valid_data`, using that dataset's exclusion registry.
- Hard negatives: filenames containing **both** `20261005` and `switch_hand`. Only the two training files are oversampled; the two validation files remain held out.
- Fine-tune every layer with a fresh optimizer at `1e-4`, up to 30 epochs. Each old training window is retained once per epoch; each new hand-switch training window appears five times in total.
- Each run has its own timestamped output directory, an exact `initial.pt` copy of the source checkpoint, `best.pt`, `last.pt`, `history.csv`, and sampling/provenance records.

The starting model is evaluated at epoch 0. `best.pt` initially holds those unchanged weights with the **expanded dataset configuration**; training must improve validation window accuracy (or lower loss at equal accuracy) to replace it. Window diagnostics are not continuous-event results. Use `evaluate.ipynb` for the final fixed-threshold event comparison.

## Imports

In [1]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import hashlib
import json
import random
import shutil
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import ConcatDataset, DataLoader, Subset

ROOT = Path.cwd().resolve()
if not (ROOT / 'tap_recognition').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'tap_recognition').is_dir(), 'Start from the repo root or main_notebooks/'
sys.path.insert(0, str(ROOT))

from tap_recognition.config import (
    DataConfig, EarlyStoppingConfig, LabelConfig, LSTMModelConfig,
    TrainConfig, TrainingConfig,
)
from tap_recognition.dataset import RecordedIMUDataset, load_trigger_events
from tap_recognition.model_lstm import CausalCNNLSTM
from train import (
    EarlyStopping, _augment_positive_windows, build_datasets,
    frame_soft_ce, window_class_from_probs,
)

def file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


## Configuration from the source checkpoint
Change the experiment controls below before running. The default source is the selected one-layer LSTM64-delay-8 checkpoint. Loading its metadata avoids accidentally inheriting the two-layer setting currently displayed in `training_lstm.ipynb`. BatchNorm and the model architecture are unchanged. This is a weights-only warm start with a new optimizer, not an optimizer-state resume.

In [2]:
BASE_CHECKPOINT = ROOT / 'testing_checkpoints/lstm64d8/best.pt'
DATA_ROOT = ROOT / 'data_05_10_26'
RUN_NAME = 'lstm64d8_hand_switch_ft'
FINE_TUNING_LR = 1e-4
FINE_TUNING_EPOCHS = 30
NEW_NEGATIVE_MULTIPLIER = 5  # Total appearances per epoch, including the original.
NEW_RECORDING_DATE = '20261005'
NEW_RECORDING_ACTION = 'switch_hand'

if not BASE_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Set BASE_CHECKPOINT to the preserved LSTM64D8 best.pt: {BASE_CHECKPOINT}')
source_sha256 = file_sha256(BASE_CHECKPOINT)
checkpoint = torch.load(BASE_CHECKPOINT, map_location='cpu', weights_only=True)
assert checkpoint.get('model_type') == 'lstm', 'The starting model must be an LSTM.'
model_kwargs = dict(checkpoint['model_config'])
model_kwargs['dilations'] = tuple(model_kwargs['dilations'])
assert (model_kwargs['lstm_hidden'], model_kwargs['lstm_layers']) == (64, 1), 'Expected LSTM64, one layer.'
LABEL_DELAY_FRAMES = int(checkpoint['positive_label_delay_frames'])
assert LABEL_DELAY_FRAMES == 8, 'Expected an 8-frame delayed-target source.'
AMPLITUDE_AUGMENTATION = bool(checkpoint['amplitude_augmentation'])

saved = checkpoint['train_config']
training_settings = dict(saved['training'])
early_stopping_settings = training_settings.pop('early_stopping')
training_settings.update(lr=FINE_TUNING_LR, epochs=FINE_TUNING_EPOCHS)
cfg = TrainConfig(
    seed=saved['seed'],
    out_dir=f'testing_checkpoints/{RUN_NAME}',
    init_checkpoint=str(BASE_CHECKPOINT.relative_to(ROOT)),
    freeze_except_last=False,
    data=DataConfig(**saved['data']),
    labels=LabelConfig(**saved['labels']),
    model=LSTMModelConfig(**model_kwargs),
    training=TrainingConfig(
        **training_settings,
        early_stopping=EarlyStoppingConfig(**early_stopping_settings),
    ),
)
cfg.data.train_dir = str((DATA_ROOT / 'train_data').relative_to(ROOT))
cfg.data.val_dir = str((DATA_ROOT / 'valid_data').relative_to(ROOT))
cfg.data.session_exclusions_file = str((DATA_ROOT / 'session_exclusions.csv').relative_to(ROOT))
assert cfg.data.mode == 'recorded'
assert cfg.training.early_stopping.monitor == 'val_window_acc', 'This notebook selects by window accuracy.'
assert isinstance(NEW_NEGATIVE_MULTIPLIER, int) and NEW_NEGATIVE_MULTIPLIER >= 1
assert 0 < FINE_TUNING_LR < saved['training']['lr']
print(f'Source epoch: {checkpoint["epoch"]}; SHA256: {source_sha256}')
print(f'Label delay: {LABEL_DELAY_FRAMES}; amplitude augmentation: {AMPLITUDE_AUGMENTATION}')
print(json.dumps(cfg.to_dict(), indent=2))


Source epoch: 36; SHA256: 7e278b3aeca5f2667079171546ebe3272bcfa9d01301b66f3a3c92191dc1efe2
Label delay: 8; amplitude augmentation: True
{
  "seed": 101,
  "device": "auto",
  "out_dir": "testing_checkpoints/lstm64d8_hand_switch_ft",
  "init_checkpoint": "testing_checkpoints/lstm64d8/best.pt",
  "freeze_except_last": false,
  "data": {
    "mode": "recorded",
    "train_dir": "data_fine_tunning/train_data",
    "val_dir": "data_fine_tunning/valid_data",
    "window_samples": 300,
    "sample_rate": 100.0,
    "highpass": true,
    "negative_windows_per_file": 20,
    "trigger_context_samples": 200,
    "exclusion_margin": 200,
    "dt_min": 0.12,
    "dt_max": 0.45,
    "session_exclusions_file": "data_fine_tunning/session_exclusions.csv",
    "synthetic_samples": 8000,
    "val_split": 0.15
  },
  "labels": {
    "sigma": 0.04,
    "peak_offset": 0.05,
    "half_frames": 10
  },
  "model": {
    "input_dim": 6,
    "num_classes": 3,
    "cnn_channels": 32,
    "lstm_hidden": 64,
    "l

## Seed setting

In [3]:
def fix_seeds(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.use_deterministic_algorithms(True)

fix_seeds(cfg.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}; seed: {cfg.seed}')


Device: cpu; seed: 101


## Recording split and negative labels
All recordings in the expanded directories are used, not just today's recordings. The new hand-switch files must have empty `.txt` sidecars: they contain no intended double-tap events. Existing positive labels and the dataset-specific exclusions are retained. The split is checked before building datasets.

In [4]:
def is_new_hand_switch(name):
    return NEW_RECORDING_DATE in name and NEW_RECORDING_ACTION in name

split_files = {}
new_files = {}
for split, directory in [('train', cfg.data.train_dir), ('valid', cfg.data.val_dir)]:
    paths = sorted(p for p in (ROOT / directory).glob('*.csv') if not p.name.endswith('.labels.csv'))
    assert paths, f'No recordings in {directory}'
    for path in paths:
        assert path.with_suffix('.txt').is_file(), f'Missing sparse labels: {path}'
    split_files[split] = paths
    new_files[split] = [p for p in paths if is_new_hand_switch(p.name)]
    assert len(new_files[split]) == 2, f'Expected two new hand-switch recordings in {split}.'
    for path in new_files[split]:
        assert not load_trigger_events(path.with_suffix('.txt')), f'Hand-switch recording must be negative: {path}'

assert not ({p.name for p in split_files['train']} & {p.name for p in split_files['valid']}), 'A recording appears in both splits.'
assert not ({file_sha256(p) for p in new_files['train']} & {file_sha256(p) for p in new_files['valid']}), 'New recording content is duplicated across splits.'
assert (ROOT / cfg.data.session_exclusions_file).is_file(), 'Dataset exclusion registry is missing.'
new_recordings = pd.DataFrame([
    {'split': split, 'file': p.name, 'sha256': file_sha256(p)}
    for split, paths in new_files.items() for p in paths
])
new_recordings


,split,file,sha256
0,train,imu_Miguel_switch_hand_20261005_093848.csv,9d38855101ab8b59c9e8ac2818f13b6d93e144c011d6b3...
1,train,imu_Miguel_switch_hand_20261005_102539.csv,d46253d638fee86119a513c6f7f1dffaebf3f8526b5f57...
2,valid,imu_Miguel_switch_hand_20261005_094008.csv,a699c2fad25a79239327f24ae214cdd9025e8e057d349d...
3,valid,imu_Miguel_switch_hand_20261005_102931.csv,5d40bf9a6df6e77a19c5d89e2549cf3b94e3c492f6bd09...


## Mixed-data training with explicit hard-negative repetition
A simple concatenation keeps every original training window once and appends four additional copies of each new training hand-switch window (`NEW_NEGATIVE_MULTIPLIER = 5`). Shuffle the resulting dataset each epoch. This is easier to inspect than sampling with replacement: no old training window is dropped. Validation is never repeated, augmented, or used for gradient updates.

`train_ds` remains the unique, unweighted dataset. `sampled_train_ds` is used only by the training loader. Sampling counts are written to `training_sampling.csv`. Use the same repetition scheme in your from-scratch control if you want to isolate the effect of warm-starting.

In [5]:
# Resolve a separate runtime configuration; saved cfg paths remain repository-relative.
runtime_cfg = TrainConfig(
    seed=cfg.seed, data=DataConfig(**cfg.to_dict()['data']), labels=cfg.labels,
)
for key in ('train_dir', 'val_dir', 'session_exclusions_file'):
    setattr(runtime_cfg.data, key, str(ROOT / getattr(runtime_cfg.data, key)))
train_ds, val_ds = build_datasets(runtime_cfg)
assert isinstance(train_ds, RecordedIMUDataset) and isinstance(val_ds, RecordedIMUDataset), 'Use the explicit recording split; no random fallback split.'
new_train_names = {p.name for p in new_files['train']}
new_val_names = {p.name for p in new_files['valid']}
new_train_indices = [i for i, meta in enumerate(train_ds.window_meta) if meta.source_file in new_train_names]
new_val_indices = [i for i, meta in enumerate(val_ds.window_meta) if meta.source_file in new_val_names]
assert {train_ds.window_meta[i].source_file for i in new_train_indices} == new_train_names
assert {val_ds.window_meta[i].source_file for i in new_val_indices} == new_val_names
for dataset, indices in [(train_ds, new_train_indices), (val_ds, new_val_indices)]:
    assert all(int(dataset[i]['window_label']) == 0 for i in indices)
    assert all(bool(torch.all(dataset[i]['frame_labels'][:, 0] == 1)) for i in indices)

extra_indices = new_train_indices * (NEW_NEGATIVE_MULTIPLIER - 1)
sampled_train_ds = ConcatDataset([train_ds, Subset(train_ds, extra_indices)]) if extra_indices else train_ds
train_loader = DataLoader(
    sampled_train_ds, batch_size=cfg.training.batch_size, shuffle=True,
    num_workers=cfg.training.num_workers, generator=torch.Generator().manual_seed(cfg.seed),
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.training.batch_size, shuffle=False, num_workers=cfg.training.num_workers,
)
new_val_loader = DataLoader(
    Subset(val_ds, new_val_indices), batch_size=cfg.training.batch_size,
    shuffle=False, num_workers=cfg.training.num_workers,
)
sampling_table = pd.DataFrame([
    {
        'window_index': i, 'source_file': meta.source_file,
        'window_class': int(train_ds[i]['window_label']),
        'new_hand_switch': meta.source_file in new_train_names,
        'appearances_per_epoch': NEW_NEGATIVE_MULTIPLIER if meta.source_file in new_train_names else 1,
    }
    for i, meta in enumerate(train_ds.window_meta)
])
assert int(sampling_table.appearances_per_epoch.sum()) == len(sampled_train_ds)
print(f'Unique train windows: {len(train_ds)}; sampled per epoch: {len(sampled_train_ds)}')
print(f'New train negatives: {len(new_train_indices)} x {NEW_NEGATIVE_MULTIPLIER}; new validation negatives: {len(new_val_indices)} x 1')
print(f'New hand-switch share of training epoch: {NEW_NEGATIVE_MULTIPLIER * len(new_train_indices) / len(sampled_train_ds):.1%}')
print(f'Train positive windows: {sum(int(train_ds[i]["window_label"]) > 0 for i in range(len(train_ds)))}')
print(f'Validation windows: {len(val_ds)}; positives: {sum(int(val_ds[i]["window_label"]) > 0 for i in range(len(val_ds)))}')
sampling_table.groupby(['new_hand_switch', 'window_class']).agg(
    unique_windows=('window_index', 'count'), per_epoch=('appearances_per_epoch', 'sum'),
)


Train dirs:
  /home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/data_fine_tunning/train_data
  Excluding session(s) [8] from imuStrict_Miguel_knock_twice_left_20260929_115133.csv
  Excluding session(s) [15] from imu_Miguel_knock_twice_left_20260924_093125.csv
  Excluding session(s) [2, 6, 12, 13, 16, 19] from imu_Miguel_knock_twice_left_20260925_105524.csv
  Excluding session(s) [1, 6, 7, 10, 17, 19] from imu_Miguel_knock_twice_left_20260925_152309.csv
  Excluding session(s) [10, 19] from imu_Miguel_knock_twice_right_20260923_113114.csv
  Excluding session(s) [2, 9] from imu_Miguel_knock_twice_right_20260924_093233.csv
  Excluding session(s) [4, 9, 13, 14, 16, 18, 19, 20] from imu_Miguel_knock_twice_right_20260925_105401.csv
  Excluding session(s) [19] from imu_Miguel_knock_twice_right_20260925_152103.csv
  Excluding session(s) [1] from imu_OLV_knock_twice_left_20260923_174506.csv
Val dirs:
  /home/aprohack/desktop/work/tap-project/project/tapRecognitionR

unique_windows  per_epoch
new_hand_switch window_class                           
False           0                        600        600
                1                        565        565
                2                        567        567
True            0                         40        200

## Load weights and create a separate run

In [6]:
model = CausalCNNLSTM(**model_kwargs).to(device)
model.load_state_dict(checkpoint['model_state'], strict=True)
assert all(p.requires_grad for p in model.parameters()), 'This experiment fine-tunes all layers.'
run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
out_dir = ROOT / cfg.out_dir / run_timestamp
assert BASE_CHECKPOINT.parent.resolve() not in out_dir.resolve().parents
out_dir.mkdir(parents=True, exist_ok=False)
shutil.copy2(BASE_CHECKPOINT, out_dir / 'initial.pt')
assert file_sha256(out_dir / 'initial.pt') == source_sha256, 'Source checkpoint changed during setup.'
sampling_table.to_csv(out_dir / 'training_sampling.csv', index=False)
new_recordings.to_csv(out_dir / 'new_recordings.csv', index=False)
fine_tuning_metadata = {
    'source_checkpoint': str(BASE_CHECKPOINT.relative_to(ROOT)),
    'source_sha256': source_sha256,
    'source_epoch': int(checkpoint['epoch']),
    'new_recording_date': NEW_RECORDING_DATE,
    'new_recording_action': NEW_RECORDING_ACTION,
    'new_negative_multiplier': NEW_NEGATIVE_MULTIPLIER,
    'new_train_files': sorted(new_train_names),
    'new_valid_files': sorted(new_val_names),
    'train_windows_unique': len(train_ds),
    'train_windows_per_epoch': len(sampled_train_ds),
    'new_train_windows_unique': len(new_train_indices),
    'validation_windows': len(val_ds),
    'exclusions_sha256': file_sha256(ROOT / cfg.data.session_exclusions_file),
    'checkpoint_selection': 'maximum validation window accuracy; lower loss breaks exact ties; includes epoch 0',
}
(out_dir / 'run_config.json').write_text(json.dumps({
    'train_config': cfg.to_dict(),
    'fine_tuning': fine_tuning_metadata,
    'positive_label_delay_frames': LABEL_DELAY_FRAMES,
    'amplitude_augmentation': AMPLITUDE_AUGMENTATION,
}, indent=2) + '\n', encoding='utf-8')
print(f'Loaded {sum(p.numel() for p in model.parameters()):,} parameters from {BASE_CHECKPOINT}')
print(f'New run: {out_dir}')


Loaded 43,139 parameters from /home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/testing_checkpoints/lstm64d8/best.pt
New run: /home/aprohack/desktop/work/tap-project/project/tapRecognitionRui/TapRecognition/testing_checkpoints/lstm64d8_hand_switch_ft/20261005_110400_612376


## Optimizer and learning rate

In [7]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=cfg.training.lr, weight_decay=cfg.training.weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.training.epochs)
cfg_es = cfg.training.early_stopping
early_stop = EarlyStopping(cfg_es)


## Delayed targets and train/validation functions
These follow `training_lstm.ipynb`. Only soft frame targets move right by eight frames; sparse annotations and window classes remain unchanged. Negative targets remain entirely `none`. Training metrics use the repeated training dataset's actual sample count.

In [8]:
def shift_event_targets_right(frame_labels, shift):
    """Move event probabilities later and assign remaining mass to none."""
    if shift < 0:
        raise ValueError('shift must be non-negative')
    if shift == 0:
        return frame_labels
    if shift >= frame_labels.shape[1]:
        raise ValueError('shift must be smaller than the number of frames')
    shifted = torch.zeros_like(frame_labels)
    shifted[..., 0] = 1.0
    shifted[:, shift:, 1:] = frame_labels[:, :-shift, 1:]
    shifted[..., 0] = 1.0 - shifted[..., 1:].sum(dim=-1)
    return shifted


In [9]:
@torch.no_grad()
def evaluate(model, loader, device, threshold, event_weight=10.0, neg_window_weight=2.0):
    model.eval()
    total_loss = 0.0
    correct_frames = total_frames = correct_windows = total_windows = 0
    tp = fp = fn = tn = 0
    for batch in loader:
        imu = batch['imu'].to(device)
        frame_labels = shift_event_targets_right(batch['frame_labels'].to(device), LABEL_DELAY_FRAMES)
        window_labels = batch['window_label'].to(device)
        logits, _ = model(imu)
        probs = torch.softmax(logits, dim=-1)
        loss = frame_soft_ce(
            logits, frame_labels, event_weight, window_label=window_labels,
            neg_window_weight=neg_window_weight,
        )
        total_loss += loss.item() * imu.size(0)
        correct_frames += (probs.argmax(dim=-1) == frame_labels.argmax(dim=-1)).sum().item()
        total_frames += frame_labels.shape[0] * frame_labels.shape[1]
        window_pred = window_class_from_probs(probs, threshold)
        correct_windows += (window_pred == window_labels).sum().item()
        total_windows += imu.size(0)
        pos_pred, pos_true = window_pred > 0, window_labels > 0
        tp += (pos_pred & pos_true).sum().item()
        fp += (pos_pred & ~pos_true).sum().item()
        fn += (~pos_pred & pos_true).sum().item()
        tn += (~pos_pred & ~pos_true).sum().item()
    return {
        'loss': total_loss / total_windows,
        'frame_acc': correct_frames / total_frames,
        'window_acc': correct_windows / total_windows,
        'precision': tp / (tp + fp + 1e-8),
        'recall': tp / (tp + fn + 1e-8),
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
    }


In [10]:
def train_one_epoch(model, loader, optimizer, device, grad_clip, threshold, event_weight, neg_window_weight=2.0):
    model.train()
    total_loss = 0.0
    correct_frames = total_frames = correct_windows = total_windows = 0
    for batch in loader:
        imu = batch['imu'].to(device)
        frame_labels = shift_event_targets_right(batch['frame_labels'].to(device), LABEL_DELAY_FRAMES)
        window_label = batch['window_label'].to(device)
        if AMPLITUDE_AUGMENTATION:
            imu = _augment_positive_windows(imu, window_label)
        logits, _ = model(imu)
        loss = frame_soft_ce(
            logits, frame_labels, event_weight, window_label=window_label,
            neg_window_weight=neg_window_weight,
        )
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        probs = torch.softmax(logits.detach(), dim=-1)
        total_loss += loss.item() * imu.size(0)
        correct_frames += (probs.argmax(dim=-1) == frame_labels.argmax(dim=-1)).sum().item()
        total_frames += frame_labels.shape[0] * frame_labels.shape[1]
        correct_windows += (window_class_from_probs(probs, threshold) == window_label).sum().item()
        total_windows += imu.size(0)
    return {
        'loss': total_loss / total_windows,
        'frame_acc': correct_frames / total_frames,
        'window_acc': correct_windows / total_windows,
    }


## Epoch 0: starting model on expanded validation
This saves the unmodified starting weights as the initial `best.pt` in the new directory, with the fine-tuning data configuration. It reports the new hand-switch subset separately at the original 0.50 threshold. Those are **negative-window false positives**, not continuous-event alarm counts. Early stopping starts from this baseline.

In [11]:
def validation_metrics(loader):
    return evaluate(
        model, loader, device, cfg.training.prediction_threshold,
        cfg.training.event_loss_weight, cfg.training.neg_window_weight,
    )

def save_checkpoint(path, epoch, val_metrics):
    torch.save({
        'model_type': 'lstm', 'model_state': model.state_dict(),
        'model_config': model_kwargs, 'train_config': cfg.to_dict(),
        'positive_label_delay_frames': LABEL_DELAY_FRAMES,
        'amplitude_augmentation': AMPLITUDE_AUGMENTATION,
        'fine_tuning': fine_tuning_metadata,
        'epoch': epoch, 'val_metrics': val_metrics,
    }, path)

baseline_metrics = validation_metrics(val_loader)
baseline_switch_metrics = validation_metrics(new_val_loader)
best_val_acc = baseline_metrics['window_acc']
best_val_loss = baseline_metrics['loss']
best_epoch = 0
early_stop.step(baseline_metrics)
save_checkpoint(out_dir / 'best.pt', 0, baseline_metrics)
history = [{
    'epoch': 0, 'lr': 0.0,
    **{f'val_{k}': v for k, v in baseline_metrics.items()},
    'val_new_switch_fp': baseline_switch_metrics['fp'],
    'val_new_switch_tn': baseline_switch_metrics['tn'],
    'val_new_switch_loss': baseline_switch_metrics['loss'],
}]
pd.DataFrame(history).to_csv(out_dir / 'history.csv', index=False)
print(f'Epoch 0 | val window acc={best_val_acc:.4f}, P={baseline_metrics["precision"]:.4f}, R={baseline_metrics["recall"]:.4f}')
print(f'New hand-switch negative windows classified as taps: {baseline_switch_metrics["fp"]}/{len(new_val_indices)}')


Epoch 0 | val window acc=0.9614, P=0.9695, R=0.9658
New hand-switch negative windows classified as taps: 3/40


## Fine-tuning loop — run when ready
This is the cell that performs gradient updates. Validation window accuracy selects checkpoints; exact accuracy ties use lower validation loss. Early stopping retains the source patience/minimum-improvement settings. The epoch-0 candidate remains available if no epoch improves. The new validation subset is diagnostic; it is not oversampled and does not add a separate loss term or stopping rule.

In [12]:
assert len(history) == 1, 'Start a fresh kernel/run before rerunning this training cell.'
stopped_epoch = 0
for epoch in range(1, cfg.training.epochs + 1):
    epoch_lr = optimizer.param_groups[0]['lr']
    train_metrics = train_one_epoch(
        model, train_loader, optimizer, device, cfg.training.grad_clip,
        cfg.training.prediction_threshold, cfg.training.event_loss_weight,
        neg_window_weight=cfg.training.neg_window_weight,
    )
    val_metrics = validation_metrics(val_loader)
    switch_metrics = validation_metrics(new_val_loader)
    scheduler.step()
    improved = val_metrics['window_acc'] > best_val_acc or (
        val_metrics['window_acc'] == best_val_acc and val_metrics['loss'] < best_val_loss
    )
    if improved:
        best_val_acc, best_val_loss = val_metrics['window_acc'], val_metrics['loss']
        best_epoch = epoch
        save_checkpoint(out_dir / 'best.pt', epoch, val_metrics)
    stopped_epoch = epoch
    save_checkpoint(out_dir / 'last.pt', epoch, val_metrics)
    history.append({
        'epoch': epoch, 'lr': epoch_lr,
        **{f'train_{k}': v for k, v in train_metrics.items()},
        **{f'val_{k}': v for k, v in val_metrics.items()},
        'val_new_switch_fp': switch_metrics['fp'],
        'val_new_switch_tn': switch_metrics['tn'],
        'val_new_switch_loss': switch_metrics['loss'],
    })
    pd.DataFrame(history).to_csv(out_dir / 'history.csv', index=False)
    print(
        f'Epoch {epoch:2d}/{cfg.training.epochs} | lr={epoch_lr:.2e} | '
        f'train loss={train_metrics["loss"]:.4f} acc={train_metrics["window_acc"]:.3f} | '
        f'val loss={val_metrics["loss"]:.4f} acc={val_metrics["window_acc"]:.3f} '
        f'P={val_metrics["precision"]:.3f} R={val_metrics["recall"]:.3f} | '
        f'new switch FP windows={switch_metrics["fp"]}/{len(new_val_indices)}'
    )
    if early_stop.step(val_metrics):
        print(f'Early stopping: no {cfg_es.monitor} improvement for {cfg_es.patience} epochs.')
        break
assert file_sha256(BASE_CHECKPOINT) == source_sha256, 'The source checkpoint changed.'
print(f'Done. Best epoch: {best_epoch}; validation window acc: {best_val_acc:.4f}; stopped at: {stopped_epoch}')
print(f'RUN_DIR for evaluate.ipynb: {out_dir.relative_to(ROOT)}')
if best_epoch == 0:
    print('No validation improvement selected: best.pt still contains the starting weights.')


Epoch  1/30 | lr=1.00e-04 | train loss=0.1578 acc=0.984 | val loss=0.2168 acc=0.959 P=0.969 R=0.960 | new switch FP windows=3/40
Epoch  2/30 | lr=9.97e-05 | train loss=0.1576 acc=0.985 | val loss=0.2183 acc=0.958 P=0.971 R=0.962 | new switch FP windows=3/40
Epoch  3/30 | lr=9.89e-05 | train loss=0.1554 acc=0.989 | val loss=0.2171 acc=0.956 P=0.961 R=0.971 | new switch FP windows=5/40
Epoch  4/30 | lr=9.76e-05 | train loss=0.1552 acc=0.989 | val loss=0.2156 acc=0.957 P=0.971 R=0.956 | new switch FP windows=3/40
Epoch  5/30 | lr=9.57e-05 | train loss=0.1550 acc=0.990 | val loss=0.2128 acc=0.958 P=0.966 R=0.964 | new switch FP windows=4/40
Epoch  6/30 | lr=9.33e-05 | train loss=0.1537 acc=0.991 | val loss=0.2139 acc=0.958 P=0.966 R=0.964 | new switch FP windows=5/40
Epoch  7/30 | lr=9.05e-05 | train loss=0.1521 acc=0.995 | val loss=0.2128 acc=0.955 P=0.958 R=0.966 | new switch FP windows=5/40
Epoch  8/30 | lr=8.72e-05 | train loss=0.1544 acc=0.989 | val loss=0.2154 acc=0.960 P=0.971 R=0.9

## Manual evaluation and three-model comparison

1. Point `RUN_DIR` in `evaluate.ipynb` at the printed fine-tuning run. Set `EVALUATION_DATA_ROOT = ROOT / 'data_05_10_26'` there and export the usual metrics.
2. Train your from-scratch control separately with one LSTM layer, hidden size 64, delay 8, amplitude augmentation enabled, source seed (101), the same expanded data/exclusions, and the same new-negative repetition scheme. Use the original learning rate (`1e-3`) for the from-scratch run.
3. Re-evaluate the **original** checkpoint on the same expanded validation data. Preserve its historical exports by using the evaluation-data override; its saved configuration still points at `data_02_10_26`.
4. Compare original, fine-tuned, and from-scratch models at **left/right 0.50/0.50, `lookahead_12f`**. Inspect hand-switch false alarms and left/right tap recall, not just overall accuracy. Keep tuned-threshold results separate.

Do not compare the old frozen export directly against these new exports: the validation fingerprint changed. All three evaluations must use the expanded dataset and identical preprocessing/matching rules. The `initial.pt` file is the exact source checkpoint; `best.pt` can remain an epoch-0 selection if fine-tuning does not improve validation. No automatic event evaluation or comparison runs in this notebook.